In [ ]:
import os
import json
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import yaml

import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from sklearn.model_selection import train_test_split

import torch
print(torch.cuda.is_available())

In [ ]:
data_path = '/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026'
train_data = os.path.join(data_path, 'train')
test_data = os.path.join(data_path, 'test')
train_data_images = os.path.join(train_data, 'train_images')
test_images_dir = os.path.join(test_data, 'test_images')

# Kaggle input paths are read-only; write everything under /kaggle/working
working_dir = '/kaggle/working'

train_images_working = os.path.join(working_dir, 'images', 'train')
train_labels_working = os.path.join(working_dir, 'labels', 'train')
valid_images_working = os.path.join(working_dir, 'images', 'val')
valid_labels_working = os.path.join(working_dir, 'labels', 'val')

for d in (train_images_working, train_labels_working, valid_images_working, valid_labels_working):
    shutil.rmtree(d, ignore_errors=True)
    os.makedirs(d, exist_ok=True)

In [ ]:
annotations_path = os.path.join(train_data, 'MAGFiLO_1.0_Annotations_kaggle2026_train.json')
with open(annotations_path) as f:
    train_labels = json.load(f)

annotations_df = pd.DataFrame(train_labels['annotations'])
images_df = pd.DataFrame(train_labels['images']).rename(columns={'id': 'image_id'})
df = annotations_df.merge(images_df, on='image_id', how='left', suffixes=('', '_image'))
df.head()

In [ ]:
def polygon_to_yolo_bbox(segmentation, width, height):
    """Axis-aligned bbox of all polygons, as normalized (xc, yc, w, h)."""
    pts = np.concatenate([np.array(poly, dtype=float).reshape(-1, 2) for poly in segmentation])
    x_min, y_min = pts.min(axis=0)
    x_max, y_max = pts.max(axis=0)
    x_min, x_max = np.clip([x_min, x_max], 0, width)
    y_min, y_max = np.clip([y_min, y_max], 0, height)
    return [
        (x_min + x_max) / 2 / width,
        (y_min + y_max) / 2 / height,
        (x_max - x_min) / width,
        (y_max - y_min) / height,
    ]


df['yolo_bbox'] = df.apply(
    lambda row: polygon_to_yolo_bbox(row['segmentation'], row['width'], row['height']),
    axis=1,
)
df['yolo_bbox'].head()

In [ ]:
all_image_files = df['file_name'].drop_duplicates()
train_files, val_files = train_test_split(all_image_files, test_size=0.2, random_state=42)

df_val = df[df['file_name'].isin(val_files)].copy()
df_train = df[df['file_name'].isin(train_files)].copy()

print(f"Train: {df_train['file_name'].nunique()} images, {len(df_train)} annotations")
print(f"Validation: {df_val['file_name'].nunique()} images, {len(df_val)} annotations")

In [ ]:
def write_split(df_split, images_out, labels_out):
    for file_name, df_image in df_split.groupby('file_name'):
        stem = Path(file_name).stem
        with open(os.path.join(labels_out, f'{stem}.txt'), 'w') as f:
            for bbox in df_image['yolo_bbox']:
                f.write('0 ' + ' '.join(f'{v:.6f}' for v in bbox) + '\n')
        shutil.copy(os.path.join(train_data_images, file_name), os.path.join(images_out, file_name))


write_split(df_train, train_images_working, train_labels_working)
write_split(df_val, valid_images_working, valid_labels_working)

print(len(os.listdir(train_images_working)), len(os.listdir(train_labels_working)))
print(len(os.listdir(valid_images_working)), len(os.listdir(valid_labels_working)))

In [ ]:
# Sanity check: draw the generated boxes on one training image
sample = df_train['file_name'].iloc[0]
image = Image.open(os.path.join(train_images_working, sample))
w, h = image.size

fig, ax = plt.subplots(1, figsize=(10, 10))
ax.imshow(image)
with open(os.path.join(train_labels_working, f'{Path(sample).stem}.txt')) as f:
    for line in f:
        _, xc, yc, bw, bh = map(float, line.split())
        ax.add_patch(patches.Rectangle(
            ((xc - bw / 2) * w, (yc - bh / 2) * h), bw * w, bh * h,
            linewidth=1, edgecolor='r', facecolor='none',
        ))
ax.axis('off')
plt.show()

In [ ]:
data = {
    'path': str(Path(working_dir).resolve()),
    'train': 'images/train',
    'val': 'images/val',
    'names': {0: 'fillament'},
}

with open(os.path.join(working_dir, 'data.yaml'), 'w', encoding='utf-8') as f:
    yaml.safe_dump(data, f, sort_keys=False)

In [ ]:
%pip install -q -U ultralytics

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo26s.pt')

In [ ]:
run_dir = os.path.join(working_dir, 'runs', 'detect', 'train')

results = model.train(
    data=os.path.join(working_dir, 'data.yaml'),
    epochs=100,
    imgsz=640,
    batch=32,
    device=0,
    project=os.path.join(working_dir, 'runs', 'detect'),
    name='train',
    exist_ok=True,
)

In [ ]:
for name in ('results.png', 'confusion_matrix.png'):
    fig, ax = plt.subplots(figsize=(20, 12), dpi=120)
    ax.imshow(plt.imread(os.path.join(run_dir, name)))
    ax.axis('off')
    fig.tight_layout(pad=0)
    plt.show()

fig, axes = plt.subplots(1, 2, figsize=(30, 14), dpi=120)
for ax, name, title in zip(axes, ('val_batch0_pred.jpg', 'val_batch0_labels.jpg'), ('Predictions', 'Ground truth')):
    ax.imshow(plt.imread(os.path.join(run_dir, name)))
    ax.set_title(title, fontsize=18)
    ax.axis('off')
fig.tight_layout(pad=0.5)
plt.show()

In [ ]:
best_weights = os.path.join(run_dir, 'weights', 'best.pt')
print(os.path.exists(test_images_dir), os.path.exists(best_weights))

best_model = YOLO(best_weights)
test_results = best_model.predict(
    source=test_images_dir,
    save=True,
    save_txt=True,
    save_conf=True,
    project=working_dir,
    name='test_predictions',
    exist_ok=True,
)

In [ ]:
from pycocotools import mask as mask_utils

# The competition scores masks, so each predicted box is submitted as a filled rectangle
rows = []

for result in test_results:
    if len(result.boxes) == 0:
        continue

    h, w = result.orig_shape
    image_stem = Path(result.path).stem

    # Highest-confidence boxes claim pixels first; later ones lose overlapping pixels
    order = result.boxes.conf.argsort(descending=True).cpu().numpy()
    xyxy = result.boxes.xyxy.cpu().numpy()
    occupied = np.zeros((h, w), dtype=bool)
    instance_number = 0

    for i in order:
        x1, y1, x2, y2 = np.round(xyxy[i]).astype(int)
        x1, x2 = np.clip([x1, x2], 0, w)
        y1, y2 = np.clip([y1, y2], 0, h)

        mask = np.zeros((h, w), dtype=bool)
        mask[y1:y2, x1:x2] = True
        mask &= ~occupied
        if not mask.any():
            continue

        occupied |= mask
        instance_number += 1

        rle = mask_utils.encode(np.asfortranarray(mask.astype(np.uint8)))
        rows.append({
            'filament_id': f'{image_stem}_{instance_number}',
            'segmentation_rle': rle['counts'].decode('utf-8'),
        })

submission = pd.DataFrame(rows, columns=['filament_id', 'segmentation_rle'])
submission

In [ ]:
SUBMISSION_CSV = os.path.join(working_dir, 'submission.csv')
submission.to_csv(SUBMISSION_CSV, index=False)

In [ ]:
import subprocess

subprocess.run(
    [
        'kaggle', 'competitions', 'submit',
        '-c', 'filament-segmentation-2026',
        '-f', SUBMISSION_CSV,
        '-m', 'Notebook submission (bbox)',
    ],
    check=True,
)